In [117]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import plotly.express as px
import plotly.graph_objects as go


# read csv files

In [118]:

cheese = pd.read_csv("cheese_data.csv")
weather = pd.read_csv("canada_weather.csv")
temp = pd.read_csv("Canada_Temperature_Data.csv")

# Transform temp dataframe to avg temp for each province for each year

In [119]:
temp["Tm"] = pd.to_numeric(temp["Tm"], errors="coerce")
prov_year = temp.groupby(["Year", "Prov"])["Tm"].mean().reset_index()
prov_year.columns = ["Year","Prov", "AvgTemp"]


Transform weather data
extract_celsius parses tempreture data

In [120]:
def extract_celsius(val):
    if pd.isna(val):
        return np.nan
    match = re.search(r"-?\d+\.?\d*", str(val))
    return float(match.group()) if match else np.nan
weather[["City", "Prov"]] = weather["Community"].str.split(", ", expand=True)
weather["Prov"] = weather["Prov"].str.strip()
cols = [
    "January(Avg. high °C (°F))",
    "January(Avg. low °C (°F))",
    "July(Avg. high °C (°F))",
    "July(Avg. low °C (°F))"
]
for c in cols:
    weather[c] = weather[c].apply(extract_celsius)
weather[cols] = weather[cols].apply(pd.to_numeric, errors="coerce")
weather["JanAvg"] = (
    weather["January(Avg. high °C (°F))"] +
    weather["January(Avg. low °C (°F))"]
) / 2

weather["JulAvg"] = (
    weather["July(Avg. high °C (°F))"] +
    weather["July(Avg. low °C (°F))"]
) / 2
weather["Prov"] = weather["Community"].str.split(",").str[-1].str.strip()

prov_weather = weather.groupby("Prov")[["JanAvg", "JulAvg"]].mean().reset_index()


Clean cheese dataframe

In [121]:
cheese = cheese.rename(columns={"ManufacturerProvCode": "Prov"})
cheese_type_temp = cheese.groupby(["Prov", "CategoryTypeEn"]).agg(
    CheeseCount=("CheeseId", "count")
).reset_index()
prov_total = cheese.groupby("Prov").agg(
    TotalCheese=("CheeseId", "count")
).reset_index()
prov_weather = weather.groupby("Prov").agg(
    JanAvg=("JanAvg", "mean"),
    JulAvg=("JulAvg", "mean")
).reset_index()
cheese_type_temp = cheese_type_temp.merge(
    prov_weather,
    on="Prov",
    how="left"
)
prov_total = prov_total.merge(
    prov_weather,
    on="Prov",
    how="left"
)

plot cheese production vs tempreture

In [122]:

prov_weather["TempRange"] = prov_weather["JulAvg"] - prov_weather["JanAvg"]
fig = go.Figure()

fig.add_trace(go.Bar(
    x=prov_weather["Prov"],
    y=prov_weather["TempRange"],
    base=prov_weather["JanAvg"],  # starts at Jan temp
    marker_color="steelblue",
    hovertemplate=(
        "Province: %{x}<br>"
        "Jan Avg: %{base}°C<br>"
        "Jul Avg: %{y + base}°C<br>"
        "Range: %{y}°C<extra></extra>"
    )
))

fig.update_layout(
    title="Temperature Range (January → July) by Province",
    yaxis_title="Temperature (°C)",
    xaxis_title="Province"
)

fig.show()

Production by province

In [123]:
prov_type["Prov_Cheese"] = prov_type["Prov"] + " | " + prov_type["CategoryTypeEn"]
fig = px.bar(
    prov_type,
    x="Prov_Cheese",
    y="CheeseCount",
    color="JanAvg",
    title="Cheese Production by Province & Type (Colored by Winter Temp)"
)

fig.show()

Are there any inferences you can make about the relationship between the weather in a province and the cheese produced?

A: After plotting cheese production against seasonal tempreture and tempreture difference, there was no difference observed between tempreture and cheese production. Quebec was a strong outlier in the plot, consistently producing highest number of cheese despite not having warmest or coldest tempreture. Noticing this, possibility of mild tempreture difference between winter and summer in Quebec was assumed to be the reason but after plotting tempreture difference vs province, it was discovered that other provinces, such as Saskatchewan, had lower tempreture difference despite having one of the lowest cheese production. 
I also paid attention to type of cheese, paying attention to the fact that some cheeses may have different production conditions. However, despite this, all cheeses were observed to be produced the most in Quebec by a large margin. One observation is that hard cheese production was low for Quebec and higher for Ontario and BC, both of which had lower janurary tempreture than Quebec. 
To conclude, all types of cheese, with exception of hard cheese, was produced the most in Quebec and did not grow in production size based on weather data such as tempreture. 

Create two different data visualizations. These can be charts or whatever you would like.

A: 2 visualizations were made, tempreture range for each province and cheese production per cheese type and province. The first plot encompasses januaray tempreture, july tempreture, average tempreture(midpoint of the two points) and range of tempreture change. Comparing the plot against (province, cheese type) vs amount plot allowed the analysis of cheese production per province based on each province's climate data.